# SaemPhysics: render a lesson preview in Google Colab

Run the cells from top to bottom. This notebook checks out the repository, installs Manim's system and Python dependencies, validates the reviewed batch and kinematics lesson, and renders a short MP4 preview. All outputs are saved in the Colab runtime's local `/content/saemphysics` folder; no Drive connection is required. Download anything you want to keep before the runtime is deleted or reset. To start from a clean runtime, choose **Runtime → Disconnect and delete runtime**, reconnect, then run the cells from the top; this removes local outputs and requires setup to run again. The full 20-video batch is an optional final step. Generated visuals are drafts; physics and release approval remain human decisions.

In [ ]:
from pathlib import Path
import os
import subprocess

REPOSITORY = "https://github.com/saemsir/physics.git"
REVISION = "main"  # For a repeatable production render, use a reviewed commit SHA.
ROOT = Path("/content/physics")

if (ROOT / ".git").is_dir():
    print(f"Updating existing checkout at {ROOT}")
elif ROOT.exists() and any(ROOT.iterdir()):
    raise RuntimeError(f"{ROOT} exists but is not a Git checkout; move or rename it, then rerun.")
else:
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(ROOT)], check=True)

subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", REVISION], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], check=True)
print(f"Repository ready at {ROOT}")

## Install render dependencies

A fresh Colab runtime needs FFmpeg/ffprobe, Cairo, Pango, and LaTeX for Manim. This setup can take several minutes. If installation fails, keep the complete cell output so the failing package or command is visible.

In [ ]:
import sys

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run([
    "apt-get", "-qq", "install", "-y",
    "ffmpeg", "libcairo2-dev", "libpango1.0-dev",
    "texlive-latex-extra", "texlive-fonts-recommended", "dvisvgm",
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt"),
], check=True)
subprocess.run(["ffmpeg", "-version"], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["ffprobe", "-version"], check=True, stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "manim", "--version"], check=True)

## Validate the model examples and compile the lesson

The batch-plan check runs the curated example problems through the physics solver and rejects mismatched expected answers. The preview uses the original constant-acceleration demo, not a copied exam question. Its text, assumptions, and expected outputs are in `data/kinematics_demo.json`.

In [ ]:
import json

BATCH_MANIFEST = ROOT / "data/video_batches/batch_001.json"
BATCH_PLAN = Path("/content/batch_plan.json")
DATABASE = Path("/content/saemphysics.sqlite")
LESSON_IR = Path("/content/kinematics-lesson.json")

subprocess.run([
    sys.executable, "-m", "content_engine", "batch-plan",
    "--manifest", str(BATCH_MANIFEST), "--output", str(BATCH_PLAN),
], cwd=ROOT, check=True)
batch_plan = json.loads(BATCH_PLAN.read_text(encoding="utf-8"))
if len(batch_plan["deliverables"]) != 20:
    raise ValueError(f"Expected 20 validated batch deliverables, found {len(batch_plan['deliverables'])}")

subprocess.run([
    sys.executable, "-m", "content_engine", "init", "--database", str(DATABASE),
], cwd=ROOT, check=True)
subprocess.run([
    sys.executable, "-m", "content_engine", "compile-lesson",
    "--database", str(DATABASE), "--problem", str(ROOT / "data/kinematics_demo.json"),
    "--output", str(LESSON_IR),
], cwd=ROOT, check=True)
lesson = json.loads(LESSON_IR.read_text(encoding="utf-8"))
print(f"Compiled: {lesson['title']}; batch deliverables validated: {len(batch_plan['deliverables'])}")

## Render one preview locally

This is the first actual Manim render. It uses a low-resolution preview, then checks that exactly one non-empty MP4 was produced and verifies its dimensions and duration. The result stays in temporary Colab storage until you download it.

In [ ]:
OUTPUT_ROOT = Path("/content/saemphysics/previews")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MEDIA_ROOT = Path("/content/physics-render")
render_env = os.environ.copy()
render_env["SAEMPHYSICS_LESSON_IR"] = str(LESSON_IR)
render_command = [
    sys.executable, "-m", "manim", "-ql",
    "--media_dir", str(MEDIA_ROOT), "--format", "mp4", "--disable_caching",
    str(ROOT / "episodes/kinematics_001/scenes.py"), "KinematicsMotion",
]
subprocess.run(render_command, cwd=ROOT, env=render_env, check=True)

renders = list(MEDIA_ROOT.rglob("KinematicsMotion.mp4"))
if len(renders) != 1 or renders[0].stat().st_size == 0:
    raise FileNotFoundError(f"Expected one non-empty KinematicsMotion.mp4, found {len(renders)}")
rendered = renders[0]
probe = subprocess.run([
    "ffprobe", "-v", "error", "-show_entries",
    "format=duration:stream=width,height", "-of", "json", str(rendered),
], check=True, capture_output=True, text=True)
video_info = json.loads(probe.stdout)
video_stream = next((stream for stream in video_info["streams"] if "width" in stream), None)
if video_stream is None or float(video_info["format"]["duration"]) <= 0:
    raise ValueError(f"ffprobe could not verify the rendered video: {video_info}")

import shutil
OUTPUT_VIDEO = OUTPUT_ROOT / "kinematics-demo-preview.mp4"
shutil.copy2(rendered, OUTPUT_VIDEO)
print(f"Verified {video_stream['width']}x{video_stream['height']} video, "
      f"{float(video_info['format']['duration']):.2f}s")
print(f"Preview saved in temporary Colab storage: {OUTPUT_VIDEO}")

In [ ]:
from IPython.display import Video, display
display(Video(str(OUTPUT_VIDEO), embed=True))

## Optional: render all 20 batch videos

Only enable this after you have watched and approved the preview. This high-quality pass produces ten vertical and ten horizontal drafts and may take a long time. The renderer writes MP4s and a resumable report with verified dimensions, durations, and file hashes to temporary Colab storage. Change `RENDER_FULL_BATCH` to `True` to run it; otherwise this cell safely skips the full batch. If rendering is interrupted but the same runtime is still available, rerun with `--resume` enabled. If you delete or reset the runtime, its local files are lost and you must rerun the notebook.

In [ ]:
RENDER_FULL_BATCH = False
if RENDER_FULL_BATCH:
    subprocess.run([
        sys.executable, str(ROOT / "tools/render_video_batch.py"),
        "--manifest", str(BATCH_MANIFEST), "--quality", "high",
        "--formats", "vertical", "horizontal", "--resume",
        "--output-dir", str(OUTPUT_ROOT.parent / "batch_001_high"),
    ], cwd=ROOT, check=True)
else:
    print("Full batch skipped. Set RENDER_FULL_BATCH = True after reviewing the preview.")

## Download outputs before ending the runtime

Run the next cell to display clickable download links for the preview and any batch MP4s/report that exist. Save the downloaded files on your computer, then upload them to Drive manually if desired. A runtime reset or deletion removes files under `/content/saemphysics`.

In [ ]:
from IPython.display import FileLink, display

if not OUTPUT_VIDEO.is_file():
    raise FileNotFoundError(f"Preview not found: {OUTPUT_VIDEO}. Run the preview cells first.")

download_paths = [OUTPUT_VIDEO]
batch_output = OUTPUT_ROOT.parent / "batch_001_high"
if batch_output.is_dir():
    download_paths.extend(sorted(batch_output.glob("*.mp4")))
    report = batch_output / "render_report.json"
    if report.is_file():
        download_paths.append(report)

print(f"{len(download_paths)} file(s) available to download:")
for path in download_paths:
    display(FileLink(str(path)))